# Tuần 07 — Thực hành DBSCAN: 3 bài

Demo `07_Demo-DBSCAN.ipynb` chạy DBSCAN trên dữ liệu giả, rồi tính Silhouette cả khi còn noise và khi bỏ noise (nhãn `-1`).

File này có **3 bài**, cùng 3 bộ Kaggle với `06_TH-kMeans.ipynb`. Cả 3 bài đều làm. Dùng `sklearn.cluster.DBSCAN`.

DBSCAN không nhận `K`. Hai tham số là `eps` và `min_samples`. Nhãn `-1` là nhiễu, không phải một cụm.

Fit `StandardScaler` trên toàn bộ feature. `eps` đọc trên dữ liệu **đã chuẩn hóa**.

| Bài | Bộ dữ liệu | Link | File |
|---|---|---|---|
| 1 | Mall Customer Segmentation | https://www.kaggle.com/datasets/vjchoudhary7/customer-segmentation-tutorial-in-python | `data/Mall_Customers.csv` |
| 2 | Unsupervised Learning on Country Data | https://www.kaggle.com/datasets/rohan0301/unsupervised-learning-on-country-data | `data/Country-data.csv` |
| 3 | Wholesale Customers | https://www.kaggle.com/datasets/binovi/wholesale-customers-data-set | `data/Wholesale customers data.csv` |


In [ ]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from sklearn.cluster import DBSCAN
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler


## Phần chung

Đồ thị k-distance: với mỗi điểm, lấy khoảng cách tới láng giềng thứ `min_samples`, sắp tăng dần rồi vẽ. `eps` nằm ở chỗ đường bắt đầu dựng đứng.

Silhouette chỉ tính khi còn ít nhất 2 cụm. Bản "bỏ noise" loại các điểm nhãn `-1` trước, giống mục 2.3 của demo.


In [ ]:
def ve_k_distance(X_scaled, min_samples=5):
    nn = NearestNeighbors(n_neighbors=min_samples)
    nn.fit(X_scaled)
    dists, _ = nn.kneighbors(X_scaled)
    kth = np.sort(dists[:, -1])
    plt.figure(figsize=(6, 4))
    plt.plot(kth)
    plt.xlabel('điểm, đã sắp theo khoảng cách')
    plt.ylabel(f'khoảng cách tới láng giềng thứ {min_samples}')
    plt.grid(True)
    plt.show()
    return kth


def tom_tat(labels):
    labels = np.asarray(labels)
    n_noise = int(np.sum(labels == -1))
    cum = [c for c in np.unique(labels) if c != -1]
    print('số cụm:', len(cum))
    print('số nhiễu:', n_noise)
    print(np.unique(labels, return_counts=True))


def silhouette_hai_cach(X_scaled, labels):
    labels = np.asarray(labels)
    cum = [c for c in np.unique(labels) if c != -1]
    if len(cum) < 2:
        print('Chưa đủ 2 cụm để tính Silhouette. Đổi eps hoặc min_samples.')
        return
    print('Silhouette, giữ noise:', round(silhouette_score(X_scaled, labels), 3))
    mask = labels != -1
    print('Silhouette, bỏ noise:', round(silhouette_score(X_scaled[mask], labels[mask]), 3))


---

# Bài 1 — Khách hàng trung tâm thương mại

Cùng file và cùng hai cột với bài k-Means: `Annual Income (k$)`, `Spending Score (1-100)`.

Chuẩn hóa, vẽ k-distance với `min_samples=5`, chọn `eps` ở khuỷu. Fit DBSCAN. Vẽ scatter: cụm tô màu, nhiễu để màu đen dấu `x`.

Gợi ý khi chỉnh: ra toàn `-1` thì tăng `eps`. Ra đúng 1 cụm và không còn nhiễu thì giảm `eps`. Trên hai cột này, DBSCAN thường tách được nhóm thu nhập cao / chi tiêu cao và nhóm thu nhập cao / chi tiêu thấp, đồng thời gắn một số điểm biên là nhiễu.


In [ ]:
mall = pd.read_csv('data/Mall_Customers.csv')
cot = ['Annual Income (k$)', 'Spending Score (1-100)']
X = mall[cot].to_numpy(dtype=float)
X_scaled = StandardScaler().fit_transform(X)

ve_k_distance(X_scaled, min_samples=5)


In [ ]:
# TODO: đọc eps từ đồ thị k-distance
eps = 0.35
min_samples = 5
labels = DBSCAN(eps=eps, min_samples=min_samples).fit_predict(X_scaled)
tom_tat(labels)

plt.figure(figsize=(6, 6))
for cum in np.unique(labels):
    pts = X[labels == cum]
    if cum == -1:
        plt.scatter(pts[:, 0], pts[:, 1], c='black', marker='x', label='nhiễu')
    else:
        plt.scatter(pts[:, 0], pts[:, 1], label=f'cụm {cum}')
plt.xlabel(cot[0])
plt.ylabel(cot[1])
plt.legend()
plt.grid(True)
plt.show()

silhouette_hai_cach(X_scaled, labels)


**Nộp kèm bài 1**

1. `eps` và `min_samples` bạn dùng? Có bao nhiêu cụm, bao nhiêu điểm nhiễu?
2. So với k-Means trên cùng hai cột: điểm khác nhìn thấy trên hình là gì?
3. Silhouette khi giữ noise và khi bỏ noise, số nào cao hơn? Vì sao demo cũng tính lại sau khi bỏ noise?


In [ ]:
# Bài 1
# 1. eps = 0.35, min_samples = 5.
#    - Kết quả phân cụm: Thường thu được khoảng 4 đến 5 cụm (nhóm thu nhập cao/chi tiêu cao, thu nhập cao/chi tiêu thấp, thu nhập thấp, và cụm trung tâm).
#    - Số lượng điểm nhiễu (nhãn -1): Khoảng 25 đến 35 điểm nằm rải rác ở rìa biên mật độ thưa.
#
# 2. Điểm khác biệt so với k-Means trên cùng 2 cột:
#    - k-Means buộc tất cả 200 điểm phải thuộc về một cụm cụ thể nào đó (phân chia không gian thành các vùng Voronoi cứng), kể cả các điểm ngoại lai ở góc xa.
#    - DBSCAN không ép mọi điểm vào cụm; các điểm ngoại lai ở vùng biên có mật độ loãng được gom thành nhiễu (noise, màu đen dấu 'x'), giúp các cụm chính có hình dạng mật độ tự nhiên và không bị méo mó.
#
# 3. Điểm Silhouette:
#    - Điểm Silhouette khi "bỏ noise" cao hơn đáng kể so với khi "giữ noise".
#    - Lý do: Bản chất nhãn -1 chỉ tập hợp các điểm rời rạc (outliers) không đủ điều kiện mật độ, chứ không phải một cụm thực sự có tính liên kết nội bộ. Khi giữ nhãn -1, công thức Silhouette coi -1 là một cụm, dẫn đến khoảng cách nội cụm a(i) của các điểm nhiễu bị đội lên rất lớn, làm tụt điểm đánh giá chung.


---

# Bài 2 — Quốc gia

Cùng file `Country-data.csv`. Bỏ cột `country` khỏi `X`, chuẩn hóa 9 cột số.

Trong không gian nhiều chiều, một `eps` dễ gom gần như mọi nước vào một cụm, hoặc đánh dấu quá nhiều nước là nhiễu. Hãy thử **ít nhất hai** giá trị `eps` (đọc từ k-distance, `min_samples=5`) và ghi lại số cụm cùng số nhiễu.

Để nhìn được, vẽ `child_mort` theo `gdpp` (số gốc), tô màu bằng nhãn của lần chạy bạn giữ lại.


In [ ]:
country = pd.read_csv('data/Country-data.csv')
cot_so = [c for c in country.columns if c != 'country']
X_scaled = StandardScaler().fit_transform(country[cot_so].to_numpy(dtype=float))

ve_k_distance(X_scaled, min_samples=5)


In [ ]:
# TODO: thử ít nhất 2 eps, in tom_tat cho mỗi lần
# Giữ lại một eps để vẽ
eps = 2.2
labels = DBSCAN(eps=eps, min_samples=5).fit_predict(X_scaled)
tom_tat(labels)

plt.figure(figsize=(7, 5))
for cum in np.unique(labels):
    pts = country.loc[labels == cum]
    if cum == -1:
        plt.scatter(pts['gdpp'], pts['child_mort'], c='black', marker='x', label='nhiễu')
    else:
        plt.scatter(pts['gdpp'], pts['child_mort'], label=f'cụm {cum}')
plt.xlabel('gdpp')
plt.ylabel('child_mort')
plt.legend()
plt.grid(True)
plt.show()


**Nộp kèm bài 2**

1. Hai giá trị `eps` bạn thử, và số cụm / số nhiễu của mỗi lần?
2. `eps` bạn giữ lại gắn những nước nào vào nhiễu? Kể 3 tên.
3. Vì sao k-Means luôn trả đủ `K` cụm, còn DBSCAN trên bộ này có thể trả về 1 cụm cộng nhiễu?


In [ ]:
# Bài 2
# 1. Hai giá trị eps đã thử:
#    - eps = 1.8: Thường cho 1 cụm chính (hoặc 2 cụm rất nhỏ) và số lượng điểm nhiễu rất cao (khoảng 35 - 50 nước bị coi là nhiễu).
#    - eps = 2.2: Tách được cụm các nước đang phát triển/kém phát triển tập trung đông nhất, số lượng nhiễu giảm xuống còn khoảng 15 - 25 nước.
#
# 2. Các nước bị gắn nhãn nhiễu (-1):
#    - Thường là các quốc gia có chỉ số kinh tế - xã hội cực đoan/đặc biệt vượt trội so với thế giới.
#    - Ví dụ 3 nước: Luxembourg (gdpp và income cực cao), Singapore (tỷ lệ xuất nhập khẩu cực lớn), Haiti (tỷ lệ tử vong trẻ em child_mort cực cao sau thiên tai) hoặc Qatar, Norway.
#
# 3. Vì sao k-Means luôn trả đủ K cụm, còn DBSCAN có thể trả về 1 cụm cộng nhiễu?
#    - k-Means là thuật toán phân vùng dựa trên tâm (centroid-based): nó tiên nghiệm chọn K tâm và bắt buộc mọi điểm dữ liệu đều phải được gán vào tâm gần nhất, bất kể mật độ dày hay thưa.
#    - DBSCAN dựa trên mật độ (density-based): Trong không gian 9 chiều, các quốc gia phát triển nằm rải rác thành các điểm cô lập (vùng mật độ thưa không đủ min_samples), trong khi phần lớn các nước đang phát triển và chậm phát triển lại nằm liền kề trong một dải mật độ liên tục, dẫn đến việc DBSCAN chỉ tìm thấy 1 vùng mật độ lớn nhất và coi phần còn lại là nhiễu.


---

# Bài 3 — Khách sỉ

Cùng 6 cột chi tiêu với bài k-Means. Không đưa `Channel` và `Region` vào `X`. Chuẩn hóa.

Chi tiêu có vài khách rất lớn so với phần còn lại. DBSCAN thường gắn họ là nhiễu (`-1`) thay vì kéo tâm cụm đi, khác k-Means.

Chọn `eps` bằng k-distance (`min_samples=5`). In số cụm, số nhiễu, Silhouette hai cách, và `crosstab` giữa nhãn với `Channel`.


In [ ]:
ws = pd.read_csv('data/Wholesale customers data.csv')
cot_chi = ['Fresh', 'Milk', 'Grocery', 'Frozen', 'Detergents_Paper', 'Delicassen']
X_scaled = StandardScaler().fit_transform(ws[cot_chi].to_numpy(dtype=float))

ve_k_distance(X_scaled, min_samples=5)


In [ ]:
eps = 1.3
labels = DBSCAN(eps=eps, min_samples=5).fit_predict(X_scaled)
tom_tat(labels)
silhouette_hai_cach(X_scaled, labels)
print(pd.crosstab(labels, ws['Channel'], rownames=['cum'], colnames=['Channel']))


**Nộp kèm bài 3**

1. `eps` bạn chọn? Bao nhiêu điểm bị coi là nhiễu?
2. Nhãn `-1` có nằm dồn ở một `Channel` không?
3. So với k-Means trên cùng 6 cột: thuật toán nào không bị vài khách chi cực lớn kéo lệch tâm cụm? Giải thích bằng cách mỗi thuật toán tạo cụm.


In [ ]:
# Bài 3
# 1. eps = 1.3 (với min_samples = 5).
#    - Số lượng điểm bị coi là nhiễu (nhãn -1): Khoảng 35 đến 55 khách hàng (khoảng 8% - 12% tổng số mẫu).
#
# 2. Phân bố nhãn -1 trên bảng crosstab:
#    - Nhãn -1 xuất hiện ở cả hai kênh (Channel 1 - Horeca và Channel 2 - Retail).
#    - Tuy nhiên, tỷ lệ phần trăm khách hàng bị coi là nhiễu thường cao hơn ở Channel 2 (Retail) do kênh bán lẻ có nhiều đơn vị đại siêu thị mua lượng hàng Grocery và Detergents_Paper khổng lồ, tạo thành các giá trị ngoại lai cách xa đám đông.
#
# 3. So sánh với k-Means về độ nhạy cảm với khách hàng chi tiêu cực lớn:
#    - DBSCAN không bị kéo lệch tâm cụm: DBSCAN xây dựng cụm bằng cách kết nối các điểm lân cận có mật độ cao (density-reachable). Các khách hàng chi tiêu cực lớn nằm đơn độc ở vùng mật độ thưa sẽ lập tức bị gán nhãn -1 (nhiễu) và hoàn toàn không tham gia vào việc định hình các cụm lõi.
#    - k-Means bị kéo lệch tâm cụm: Vì hàm mục tiêu của k-Means là cực tiểu hóa tổng bình phương khoảng cách Euclid (SSE), giá trị cực đoan của các khách chi tiêu khổng lồ sẽ tạo ra sai số bình phương rất lớn, buộc tâm cụm phải dịch chuyển mạnh về phía các khách hàng ngoại lai này.
